# 02 · Data conversion

Turns the raw datasets into one paired layout that Ultralytics can read: two folders of images with identical file names (`images/visible/...`, `images/infrared/...`), YOLO labels, and three data YAMLs per dataset (paired 4-channel, visible-only, thermal-only). Images are stored with their long side at 640 px, the resolution the models train at, which also makes loading fast. Every choice is written into a `dataset_card.json`.

| | |
|---|---|
| **Kaggle settings** | Accelerator **GPU T4 x2**, Internet **on** |
| **Inputs to attach** | `cffm-net-pilot`, raw LLVIP and M3FD |
| **Expected runtime** | about 5 minutes |
| **Produces** | `data/llvip/`, `data/m3fd/` (and, if you save a version, an attachable output) |

In [ ]:
# --------------------------------------------------------------------------------------------
# Setup: the same cell opens every notebook. It finds the code, installs it, and checks the GPU.
#
# On Kaggle the uploaded cffm-net-pilot.zip appears, already unzipped, somewhere under
# /kaggle/input. Inputs are read-only, so we copy the project to /kaggle/working and install
# it from there ("pip install -e" makes `import cffm` work in this kernel, in DataLoader
# workers, and in the extra processes Ultralytics starts when it trains on two GPUs).
# Locally, notebooks live in cffm-net-pilot/notebooks/<phase>/; the project is the first folder up
# that contains src/cffm.
# --------------------------------------------------------------------------------------------
import glob, shutil, subprocess, sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    # up to five levels deep: /kaggle/input/<slug>/, /kaggle/input/<slug>/cffm-net-pilot/,
    # /kaggle/input/datasets/<owner>/<slug>/..., whichever layout this Kaggle version uses
    hits = [Path(p).parent for k in range(1, 6) for p in glob.glob("/kaggle/input" + "/*" * k + "/pyproject.toml")
            if (Path(p).parent / "src" / "cffm").is_dir()]
    assert hits, "Add the 'cffm-net-pilot' dataset (the uploaded zip) as an input to this notebook."
    PROJECT = Path("/kaggle/working/cffm-net-pilot")
    if not PROJECT.exists():
        shutil.copytree(hits[0], PROJECT, copy_function=shutil.copyfile)
        for p in [PROJECT, *PROJECT.rglob("*")]:     # the input was read-only; our copy must be writable
            p.chmod(p.stat().st_mode | 0o200)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.171",
                    "faster-coco-eval>=1.6.7", "cloudpickle"], check=True)   # cloudpickle: two-GPU launcher
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(PROJECT)], check=True)
else:
    PROJECT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "cffm").is_dir())
sys.path.insert(0, str(PROJECT / "src"))

import cffm
from cffm import env as cenv, pipeline
print("cffm", cffm.__version__, "imported from", Path(cffm.__file__).parent)   # must be inside PROJECT
env = cenv.setup()                       # prints platform, GPUs and paths; moves into the project folder
plan = pipeline.load_plan(env.project)   # configs/pilot.yaml: every run and its settings

## Convert

The options (which LLVIP frames to keep, the M3FD split seed) come from `configs/pilot.yaml`, so this notebook and every training notebook make the identical dataset.

In [ ]:
cards = {}
for name in ("llvip", "m3fd"):
    out = pipeline.prepare(name, env, plan, force=False)
    cards[name] = __import__("json").loads((out / "dataset_card.json").read_text())
    print(name, "->", out)

## What we made

The object-size histogram is the first look at how small these objects really are at the resolution the network sees.

In [ ]:
import pandas as pd
rows = [{"dataset": n, "train": c["images"]["train"], "val (full)": c["images"]["val"], "objects": c["objects"],
         **c["object_size_bins"]} for n, c in cards.items()]
pd.DataFrame(rows).set_index("dataset")

In [ ]:
print((env.data / "llvip" / "data_paired.yaml").read_text())

## Sanity checks

Every visible image has a thermal partner of the same size and a label file; every box lies inside the image.

In [ ]:
import cv2, numpy as np
for name in ("llvip", "m3fd"):
    root = env.data / name
    for split in ("train", "val"):
        vis = sorted((root / "images" / "visible" / split).glob("*.jpg"))
        missing = [v for v in vis if not (root / "images" / "infrared" / split / v.name).exists()
                   or not (root / "labels" / "visible" / split / (v.stem + ".txt")).exists()]
        boxes = np.array([list(map(float, l.split()[1:])) for f in (root / "labels" / "visible" / split).glob("*.txt")
                          for l in f.read_text().splitlines() if l.strip()], dtype=float).reshape(-1, 4)
        inside = ((boxes[:, :2] - boxes[:, 2:] / 2 >= -1e-6) & (boxes[:, :2] + boxes[:, 2:] / 2 <= 1 + 1e-6)).all()
        # sizes: reading every image takes minutes, every 25th pair is plenty to catch a converter bug
        size_ok = all(cv2.imread(str(v)).shape[:2] == cv2.imread(str(root / "images" / "infrared" / split / v.name)).shape[:2]
                      for v in vis[::25])
        print(f"{name} {split}: {len(vis)} pairs, {len(missing)} incomplete, {len(boxes)} boxes, "
              f"all inside image: {bool(inside)}, pair sizes match: {size_ok}")
        assert vis and not missing and inside and size_ok

## Look at a few converted pairs, with their labels

Green boxes on both images. If a box sits on a person in the visible image but beside them in the thermal one, the dataset is misregistered (notebook 03 measures this properly).

In [ ]:
import matplotlib.pyplot as plt
from cffm.data import read_pair
from cffm.viz import draw_pair

def show(name, k=3):
    root = env.data / name
    vis = sorted((root / "images" / "visible" / "val").glob("*.jpg"))
    vis = vis[:: max(1, len(vis) // k)][:k]                      # k pairs spread over the split
    names = cards[name]["names"]
    for v in vis:
        pair = read_pair(str(v)); h, w = pair.shape[:2]
        gt = []
        for line in (root / "labels" / "visible" / "val" / (v.stem + ".txt")).read_text().splitlines():
            c, x, y, bw, bh = map(float, line.split())
            gt.append([(x - bw / 2) * w, (y - bh / 2) * h, (x + bw / 2) * w, (y + bh / 2) * h, 1.0, c])
        plt.figure(figsize=(14, 4.5)); plt.imshow(draw_pair(pair, gt, names)); plt.axis("off"); plt.title(f"{name}: {v.name}")
        plt.show()

show("llvip"); show("m3fd")

### Keep the results

On Kaggle, click **Save Version → Save & Run All (Commit)**. The notebook then reruns top to bottom in the
background, and its `runs/` folder becomes an output that notebooks **09** and **15** can attach as an input
(*Add Input → Your Work → this notebook*). Without a saved version, the checkpoints vanish when the session ends.